Load data

In [1]:
!pip install huggingface_hub
!pip install transformers datasets tokenizers seqeval -q


In [1]:
from utilities import preprocess, load_data

label_list=load_data('uk')[1]
label_list

['B-EVT',
 'B-LOC',
 'B-ORG',
 'B-PER',
 'B-PRO',
 'I-EVT',
 'I-LOC',
 'I-ORG',
 'I-PER',
 'I-PRO',
 'O']

In [2]:

data=load_data('bg')
raw_dataset, label_list, label2id, id2label= data[0], data[1],data[2], data[3]

In [4]:

data=load_data('sl')
raw_dataset, label_list, label2id, id2label= data[0], data[1],data[2], data[3]
print(raw_dataset["test"][0])

{'tokens': ['Obtoženka', 'bogokletstvo', 'Asia', 'Bibi', 'zapustiti', 'Pakistan', '.'], 'ner_tags': [10, 10, 3, 8, 10, 1, 10]}


In [5]:

data=load_data('sl', cyrillic=True)
raw_dataset, label_list, label2id, id2label= data[0], data[1],data[2], data[3]
print(raw_dataset["test"][0])

TypeError: load_data() got an unexpected keyword argument 'cyrillic'

In [16]:
from utilities import preprocess
# get data
model_name="xlm-roberta-large"
data=preprocess(language_code='ru', model_name=model_name)
tokenized_datasets_ru, label_list, label2id, id2label, tokenizer= data[0], data[1],data[2], data[3], data[4]
print(tokenized_datasets_ru["train"][0])


Map:   0%|          | 0/7560 [00:00<?, ? examples/s]

Map:   0%|          | 0/5152 [00:00<?, ? examples/s]

Map:   0%|          | 0/10692 [00:00<?, ? examples/s]

{'tokens': ['В', 'Пакистан', 'протестовать', 'против', 'отмена', 'приговор', 'за', 'богохульство', '.'], 'ner_tags': [10, 1, 10, 10, 10, 10, 10, 10, 10], 'input_ids': [0, 417, 174222, 34800, 27224, 4988, 183, 27145, 151609, 61, 85063, 244, 29524, 3280, 6, 5, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 

In [24]:

# intitialize model
from transformers import AutoTokenizer,AutoModelForTokenClassification,AutoModelForTokenClassification, AutoConfig
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForTokenClassification
from transformers import pipeline

config = AutoConfig.from_pretrained(model_name, num_labels=len(label_list) , id2label=id2label, label2id=label2id)
model = AutoModelForTokenClassification.from_config(config)
data_collator = DataCollatorForTokenClassification(tokenizer) 
args = TrainingArguments(
"test-ner",
evaluation_strategy = "epoch",
learning_rate=2e-5,
per_device_train_batch_size=16,
per_device_eval_batch_size=16,
num_train_epochs=3,
weight_decay=0.01,
)


C:\Users\maryz\AppData\Roaming\Python\Python311\site-packages\transformers\training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [25]:
# train

trainer = Trainer(
    model,
    args,
   train_dataset=tokenized_datasets_ru["train"],
   eval_dataset=tokenized_datasets_ru["validation"],
   data_collator=data_collator,
   tokenizer=tokenizer
)
trainer.train()

C:\Users\maryz\AppData\Local\Temp\ipykernel_15520\2396213719.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:


# save prediction
predictions=trainer.predict(tokenized_datasets_ru["test"])
final_predictions_test = [
    [id2label[p] for p in sentence] for sentence in predictions
]

iob_predictions=[]
for sent_idx, final_pred in enumerate(final_predictions_test):

    tokens = tokenized_datasets_ru["test"]["tokens"][sent_idx]  # Tokens from your dataset
    tokens_ids = tokenized_datasets_ru["test"]["input_ids"][sent_idx]  # Tokens from your dataset
    sentence_iob = []
    pred_label_cleaned=final_pred[1:len(tokens)+1]
    for token_idx, token in enumerate(tokens):
        pred_label = pred_label_cleaned[token_idx]  # Get the label from final_predictions
        sentence_iob.append(f"{token_idx+1}\t{tokens[token_idx]}\t{pred_label}")

    iob_predictions.append(sentence_iob)

with open("ner_predictions.iob", "w") as f:
    for sentence in iob_predictions:
        for line in sentence:
            f.write(f"{line}\n")
        f.write("\n")  # Separate sentences with a blank line
#save models

from huggingface_hub import HfApi

trainer.save_model("./my_ner_model")
tokenizer.save_pretrained("./my_ner_model")
# Load your model and tokenizer
model = AutoModelForTokenClassification.from_pretrained("./my_ner_model")
tokenizer = AutoTokenizer.from_pretrained("./my_ner_model")

# Push to the hub
model.push_to_hub("your-username/your-model-name")
tokenizer.push_to_hub("your-username/your-model-name")


C:\Users\maryz\AppData\Local\Temp\ipykernel_20868\3589077742.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


ValueError: text input must be of type `str` (single example), `List[str]` (batch or single pretokenized example) or `List[List[str]]` (batch of pretokenized examples).